In [1]:
!pip install streamlit pyngrok google-genai requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 53.2 MB/s eta 0:00:00


In [2]:
def search_jobs(query, source="default"):
    return [
        {"title": f"{query} Engineer", "company": source.upper(), "skills": ["python", "ml"]}
    ]

def unified_job_search(query):
    results = []
    results.extend(search_jobs(query, "naukri"))
    results.extend(search_jobs(query, "linkedin"))
    results.extend(search_jobs(query, "indeed"))
    return results

In [3]:
%%writefile app.py
import streamlit as st
from agent import job_agent

st.title("AI Job Search Assistant")

query = st.text_input("Enter job role (example: python developer)")

if st.button("Search Jobs"):

    jobs = job_agent(query)

    if not jobs:
        st.write("No jobs found")

    else:
        for job in jobs:
            st.subheader(job["title"])
            st.write("Company:", job["company"])
            st.write("Location:", job["location"])
            st.write("Apply:", job["url"])
            st.write("---")

Writing app.py


In [4]:
!streamlit run app.py &>/dev/null &

In [5]:
!ngrok config add-authtoken 3AXWdLYFp21qNmp7Va6dBB4ZRoC_35rWntc9adaGaNCuBr9GZ

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [6]:
!streamlit run app.py &>/dev/null &

In [7]:
!killall ngrok
!killall streamlit

ngrok: no process found


In [8]:
!streamlit run app.py &>/dev/null &

# Task
Terminate all existing `ngrok` processes and then connect a new `ngrok` tunnel to expose the Streamlit application running on port 8501.

## Terminate Ngrok Processes Forcefully

### Subtask:
Terminate all ngrok related processes to ensure no conflicts when starting a new tunnel.


**Reasoning**:
The subtask requires terminating all ngrok processes to prevent conflicts. The instruction explicitly states to use `!pkill -f ngrok` for this purpose.



In [9]:
!pkill -f ngrok
print("Attempted to terminate all ngrok processes.")

Attempted to terminate all ngrok processes.


## Connect Ngrok Tunnel

### Subtask:
Attempt to establish a new `ngrok` tunnel to expose the Streamlit application.


## Summary:

### Data Analysis Key Findings
- The command `pkill -f ngrok` was executed to forcefully terminate any running `ngrok` processes.
- The execution of the termination command was confirmed by the message: "Attempted to terminate all ngrok processes."
- No errors were reported during the termination process, indicating a successful attempt to clear existing `ngrok` instances.

### Insights or Next Steps
- Successfully terminating existing `ngrok` processes is a crucial prerequisite for establishing a new, clean `ngrok` tunnel without conflicts.


# Task
Connect a new ngrok tunnel to expose the Streamlit application.

## Terminate Ngrok Processes Forcefully

### Subtask:
Increase the delay after forcefully terminating `streamlit` and `ngrok` processes to allow the `ngrok` cloud service sufficient time to release the endpoint.


**Reasoning**:
The subtask requires increasing the delay after terminating Streamlit and ngrok processes. This code block combines all the necessary commands, including importing `time`, forcefully killing `streamlit` and `ngrok` processes, calling `ngrok.kill()`, and then introducing a 30-second delay.



In [10]:
import time

!killall -9 streamlit
!pkill -f ngrok
from pyngrok import ngrok

ngrok.kill()
time.sleep(30) # Increased delay to ensure ngrok releases the endpoint
print("Streamlit and ngrok processes terminated, and a 30-second delay introduced.")

Streamlit and ngrok processes terminated, and a 30-second delay introduced.


In [11]:
!mkdir -p tools

In [12]:
%%writefile tools/job_search.py
def search_jobs(query, source="default"):
    return [
        {"title": f"{query} Engineer", "company": source.upper(), "skills": ["python", "ml"]}
    ]

def unified_job_search(query):
    results = []
    results.extend(search_jobs(query, "naukri"))
    results.extend(search_jobs(query, "linkedin"))
    results.extend(search_jobs(query, "indeed"))
    return results
def search_jobs(role_title: str) -> list:
    """
    Searches for open job postings.
    Returns a list of dictionaries containing job details.
    """
    print(f"🔧 Tool Execution: Searching for {role_title} jobs...")

    # Mock API response formatted for your Streamlit UI
    return [
        {
            "title": f"Senior {role_title.title()}",
            "company": "Tech Innovators Inc.",
            "location": "Remote",
            "url": "https://example.com/apply/1"
        },
        {
            "title": f"{role_title.title()} II",
            "company": "DataCorp Solutions",
            "location": "New York, NY",
            "url": "https://example.com/apply/2"
        }
    ]

Writing tools/job_search.py


In [13]:
%%writefile tools/company_research.py
def research_company(company_name: str) -> str:
    """
    Gathers background information on a specific company.
    """
    print(f"🔧 Tool Execution: Researching {company_name}...")

    # Mock company database/search response
    company_database = {
        "Tech Innovators Inc.": "A fast-growing startup known for its great work-life balance and AI products.",
        "DataCorp Solutions": "An established enterprise data firm with excellent healthcare benefits."
    }

    return company_database.get(company_name, "A solid company in its industry.")

Writing tools/company_research.py


In [14]:
from tools.job_search import search_jobs
from tools.company_research import research_company

In [15]:
# 1. Install necessary libraries (just in case your Colab runtime restarted)
!pip install streamlit pyngrok google-genai -q

import time
import subprocess
from pyngrok import ngrok

# 2. Set your ngrok authtoken programmatically for pyngrok
# Replace '3AXWdLYFp21qNmp7Va6dBB4ZRoC_35rWntc9adaGaNCuBr9GZ' with your actual ngrok v2 authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("3AXWdLYFp21qNmp7Va6dBB4ZRoC_35rWntc9adaGaNCuBr9GZ")

# 3. Terminate any old processes so they don't block the port
!killall -9 streamlit
!pkill -f ngrok
time.sleep(2)

# 4. Start the Streamlit app in the background
print("Starting Streamlit server...")
subprocess.Popen(["streamlit", "run", "app.py"])

# Give Streamlit a few seconds to fully boot up
time.sleep(5)

# 5. Connect the ngrok tunnel to port 8501 (Streamlit's default port)
public_url = ngrok.connect(8501)
print(f"🚀 Your AI Job Agent is live! Click this link to open it: {public_url}")

streamlit: no process found
Starting Streamlit server...
🚀 Your AI Job Agent is live! Click this link to open it: NgrokTunnel: "https://unmonarchical-unlicentiously-marquerite.ngrok-free.dev" -> "http://localhost:8501"


## Connect Ngrok Tunnel

### Subtask:
Attempt to establish a new ngrok tunnel after the extended delay to expose the Streamlit application.


## Summary:

### Data Analysis Key Findings
*   The `streamlit` process was not running when the termination command (`killall -9 streamlit`) was executed.
*   `ngrok` processes were successfully terminated using `pkill -f ngrok` and `ngrok.kill()`.
*   A 30-second delay was implemented after process termination to ensure the `ngrok` cloud service had sufficient time to release endpoints.

### Insights or Next Steps
*   The increased 30-second delay after terminating `streamlit` and `ngrok` processes is a critical step to prevent conflicts and ensure a clean state for establishing a new `ngrok` tunnel.
*   The next logical step is to attempt to establish the new `ngrok` tunnel, as the environment has been prepared by ensuring all previous related processes are terminated and endpoints are likely released.


In [16]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.6 MB/s eta 0:00:00


In [17]:
%%writefile app.py
import streamlit as st
import PyPDF2
from agent import job_agent # Keeping your existing agent import

st.title("AI Job Search Assistant & Resume Parser")

# --- RESUME PARSING SECTION ---
st.header("1. Upload Resume")
uploaded_file = st.file_uploader("Upload your resume (PDF format)", type=["pdf"])
resume_text = ""

if uploaded_file is not None:
    try:
        # Read the PDF file
        pdf_reader = PyPDF2.PdfReader(uploaded_file)
        for page in pdf_reader.pages:
            resume_text += page.extract_text() + "\n"

        st.success("Resume uploaded and read successfully!")

        with st.expander("View Extracted Text Preview"):
            st.text(resume_text[:1000] + "...\n\n(Text truncated for preview)")

        # Optional: Here you could pass `resume_text` to your Gemini agent
        # to automatically extract key skills or suggest a job role!
        # Example: extracted_skills = agent.extract_skills(resume_text)

    except Exception as e:
        st.error(f"Error reading PDF: {e}")

st.write("---")

# --- JOB SEARCH SECTION ---
st.header("2. Search for Jobs")

# If a resume is uploaded, you could potentially auto-fill this,
# but for now we'll keep the manual input as well.
query = st.text_input("Enter job role (example: python developer)")

if st.button("Search Jobs"):
    if not query:
        st.warning("Please enter a job role to search.")
    else:
        with st.spinner(f"Searching for {query} roles..."):
            jobs = job_agent(query)

            if not jobs:
                st.write("No jobs found. Try adjusting your search term.")
            else:
                for job in jobs:
                    st.subheader(job.get("title", "Unknown Title"))
                    st.write("**Company:**", job.get("company", "Unknown"))
                    st.write("**Location:**", job.get("location", "Unknown"))
                    st.markdown(f"[Apply Here]({job.get('url', '#')})")
                    st.write("---")

Overwriting app.py


In [18]:
!pip install PyPDF2 langgraph langchain openai

In [19]:
from langchain.tools import tool

@tool
def job_search_tool(query: str):
    """Search jobs from multiple platforms based on query"""
    return unified_job_search(query)

@tool
def skill_extractor_tool(text: str):
    """Extract skills from resume text"""
    return extract_skills(text)

@tool
def job_matcher_tool(input_data: dict):
    """Match jobs with user skills and return ranked results"""
    return match_jobs(input_data["skills"], input_data["jobs"])

In [20]:
from typing import TypedDict, List

class AgentState(TypedDict):
    query: str
    resume_text: str
    skills: List[str]
    jobs: List[dict]
    matched_jobs: List[dict]

In [21]:
def extract_skills_node(state):
    skills = extract_skills(state["resume_text"])
    return {"skills": skills}

def search_jobs_node(state):
    jobs = unified_job_search(state["query"])
    return {"jobs": jobs}

def match_jobs_node(state):
    matched = match_jobs(state["skills"], state["jobs"])
    return {"matched_jobs": matched}

In [22]:
from langgraph.graph import StateGraph

builder = StateGraph(AgentState)

builder.add_node("extract_skills", extract_skills_node)
builder.add_node("search_jobs", search_jobs_node)
builder.add_node("match_jobs", match_jobs_node)

builder.set_entry_point("extract_skills")

builder.add_edge("extract_skills", "search_jobs")
builder.add_edge("search_jobs", "match_jobs")

graph = builder.compile()

In [23]:
def extract_skills(text):
    skills_list = ["python", "java", "ml", "ai", "sql", "data science"]
    found_skills = [skill for skill in skills_list if skill in text.lower()]
    return found_skills

In [24]:
def search_jobs(query, source="default"):
    return [
        {
            "title": f"{query} Engineer",
            "company": source.upper(),
            "skills": ["python", "ml", "sql", "data science"]
        }
    ]

In [25]:
print(unified_job_search("Data Scientist"))

[{'title': 'Data Scientist Engineer', 'company': 'NAUKRI', 'skills': ['python', 'ml', 'sql', 'data science']}, {'title': 'Data Scientist Engineer', 'company': 'LINKEDIN', 'skills': ['python', 'ml', 'sql', 'data science']}, {'title': 'Data Scientist Engineer', 'company': 'INDEED', 'skills': ['python', 'ml', 'sql', 'data science']}]


In [26]:
def search_jobs(query, source="default"):
    return [
        {"title": f"{query} Engineer", "company": source.upper(), "skills": ["python", "ml"]}
    ]

def unified_job_search(query):
    results = []
    results.extend(search_jobs(query, "naukri"))
    results.extend(search_jobs(query, "linkedin"))
    results.extend(search_jobs(query, "indeed"))
    return results

In [27]:
def extract_skills(text):
    skills_list = ["python", "java", "ml", "ai", "sql", "data science"]
    return [skill for skill in skills_list if skill in text.lower()]

In [28]:
def match_jobs(user_skills, jobs):
    matched = []

    for job in jobs:
        score = len(set(user_skills) & set(job["skills"]))

        if score > 0:
            job["match_score"] = score
            matched.append(job)

    return sorted(matched, key=lambda x: x["match_score"], reverse=True)

In [29]:
result = graph.invoke({
    "query": "Data Scientist",
    "resume_text": "I know Python, ML, and Data Science"
})

print(result["matched_jobs"])

[{'title': 'Data Scientist Engineer', 'company': 'NAUKRI', 'skills': ['python', 'ml'], 'match_score': 2}, {'title': 'Data Scientist Engineer', 'company': 'LINKEDIN', 'skills': ['python', 'ml'], 'match_score': 2}, {'title': 'Data Scientist Engineer', 'company': 'INDEED', 'skills': ['python', 'ml'], 'match_score': 2}]


In [30]:
def interview_prep(role):
    questions = {
        "data scientist": [
            "What is overfitting?",
            "Explain bias vs variance",
            "What is a confusion matrix?"
        ]
    }
    return questions.get(role.lower(), ["Tell me about yourself"])

In [31]:
def salary_estimator(role):
    salary_data = {
        "data scientist": "₹6 LPA – ₹20 LPA",
        "software engineer": "₹4 LPA – ₹15 LPA"
    }
    return salary_data.get(role.lower(), "Not available")

In [32]:
def career_recommendation(skills):
    if "ml" in skills:
        return "Recommended Role: Data Scientist"
    elif "java" in skills:
        return "Recommended Role: Backend Developer"
    else:
        return "Recommended Role: Software Engineer"

In [33]:
def networking_tips():
    return [
        "Build LinkedIn profile",
        "Attend tech meetups",
        "Connect with professionals",
        "Participate in hackathons"
    ]

In [34]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 88.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [35]:
import spacy
nlp = spacy.load("en_core_web_sm")

def advanced_resume_analysis(text):
    doc = nlp(text)

    skills = []
    for token in doc:
        if token.pos_ == "NOUN":
            skills.append(token.text.lower())

    return list(set(skills))

In [36]:
applications = []

def apply_job(job):
    job["status"] = "Applied"
    applications.append(job)
    return "Application submitted"

def view_applications():
    return applications

In [37]:
# Example usage

skills = ["python", "ml"]

print(career_recommendation(skills))
print(salary_estimator("data scientist"))
print(interview_prep("data scientist"))
print(networking_tips())

# Apply to a job
job = result["matched_jobs"][0]
print(apply_job(job))

print(view_applications())

Recommended Role: Data Scientist
₹6 LPA – ₹20 LPA
['What is overfitting?', 'Explain bias vs variance', 'What is a confusion matrix?']
['Build LinkedIn profile', 'Attend tech meetups', 'Connect with professionals', 'Participate in hackathons']
Application submitted
[{'title': 'Data Scientist Engineer', 'company': 'NAUKRI', 'skills': ['python', 'ml'], 'match_score': 2, 'status': 'Applied'}]


In [38]:
def search_jobs(query, source="default"):
    return [
        {
            "title": f"{query} Engineer",
            "company": source.upper(),
            "skills": ["python", "ml", "sql", "data science"]
        }
    ]

In [39]:
query = input("Enter job role: ")
resume = input("Enter your skills: ")

result = graph.invoke({
    "query": query,
    "resume_text": resume
})

print("\n--- JOB MATCHES ---")
for job in result["matched_jobs"]:
    print(f"{job['title']} at {job['company']} | Score: {job['match_score']}")

skills = extract_skills(resume)

print("\n--- CAREER SUPPORT ---")
print(career_recommendation(skills))
print("Salary:", salary_estimator(query))
print("Interview Questions:", interview_prep(query))
print("Networking Tips:", networking_tips())

Enter job role: data scientist
Enter your skills: sql

--- JOB MATCHES ---
data scientist Engineer at NAUKRI | Score: 1
data scientist Engineer at LINKEDIN | Score: 1
data scientist Engineer at INDEED | Score: 1

--- CAREER SUPPORT ---
Recommended Role: Software Engineer
Salary: ₹6 LPA – ₹20 LPA
Interview Questions: ['What is overfitting?', 'Explain bias vs variance', 'What is a confusion matrix?']
Networking Tips: ['Build LinkedIn profile', 'Attend tech meetups', 'Connect with professionals', 'Participate in hackathons']


In [40]:
applications = []

def apply_job(job):
    job["status"] = "Applied"
    job["date"] = "2026-03-20"
    job["stage"] = "Resume Sent"
    applications.append(job)
    return "Application submitted"

In [41]:
def update_status(index, stage):
    applications[index]["stage"] = stage
    return applications[index]

In [42]:
def view_applications():
    for app in applications:
        print(f"{app['title']} at {app['company']} | {app['stage']}")

In [43]:
user_profile = {}

def create_profile(name, skills, goals):
    user_profile["name"] = name
    user_profile["skills"] = skills
    user_profile["goals"] = goals
    return user_profile

In [44]:
profile = create_profile(
    "Rishika",
    ["python", "ml"],
    "Become Data Scientist"
)

print(profile)

{'name': 'Rishika', 'skills': ['python', 'ml'], 'goals': 'Become Data Scientist'}


In [45]:
def personalized_jobs(profile):
    return unified_job_search(profile["goals"])

In [46]:
def job_alert(query):
    jobs = unified_job_search(query)

    print("🔔 New Job Alerts:")
    for job in jobs:
        print(f"{job['title']} at {job['company']}")

In [47]:
def search_jobs(query, source="default"):
    return [
        {
            "title": f"{query} Engineer",
            "company": source.upper(),
            "skills": ["python", "ml", "sql", "data science"]
        }
    ]

def unified_job_search(query):
    results = []
    results.extend(search_jobs(query, "naukri"))
    results.extend(search_jobs(query, "linkedin"))
    results.extend(search_jobs(query, "indeed"))
    return results

def job_alert(query):
    jobs = unified_job_search(query)

    print("🔔 New Job Alerts:")
    for job in jobs:
        print(f"{job['title']} at {job['company']}")

job_alert("Data Scientist")

🔔 New Job Alerts:
Data Scientist Engineer at NAUKRI
Data Scientist Engineer at LINKEDIN
Data Scientist Engineer at INDEED


In [48]:
import time

def auto_alert(query):
    while True:
        job_alert(query)
        time.sleep(10)  # every 10 seconds

In [49]:
profile = create_profile("Rishika", ["python", "ml"], "Data Scientist")

jobs = personalized_jobs(profile)

job = jobs[0]
apply_job(job)

view_applications()

job_alert(profile["goals"])

Data Scientist Engineer at NAUKRI | Resume Sent
🔔 New Job Alerts:
Data Scientist Engineer at NAUKRI
Data Scientist Engineer at LINKEDIN
Data Scientist Engineer at INDEED


In [50]:
import smtplib
from email.mime.text import MIMEText

def send_email_alert(message):
    sender = "your_email@gmail.com"
    receiver = "your_email@gmail.com"
    password = "your_app_password"  # NOT normal password

    msg = MIMEText(message)
    msg["Subject"] = "Job Alert 🚀"
    msg["From"] = sender
    msg["To"] = receiver

    server = smtplib.SMTP("smtp.gmail.com", 587)
    server.starttls()
    server.login(sender, password)
    server.send_message(msg)
    server.quit()

    return "Email sent!"

In [51]:
def send_notification(message):
    print("🔔 JOB ALERT 🔔")
    print(message)
    print("✅ Notification Sent Successfully!")

In [52]:
jobs = unified_job_search("Data Scientist")

message = "\n".join([f"{job['title']} at {job['company']}" for job in jobs])

send_notification(message)

🔔 JOB ALERT 🔔
Data Scientist Engineer at NAUKRI
Data Scientist Engineer at LINKEDIN
Data Scientist Engineer at INDEED
✅ Notification Sent Successfully!


In [53]:
def analyze_skills_gap(user_skills, job_skills):
    """Compares user skills against job requirements to find gaps."""
    user_set = set(s.lower() for s in user_skills)
    job_set = set(s.lower() for s in job_skills)

    # Identify skills present in the job but missing from the user profile
    missing_skills = list(job_set - user_set)

    # Custom recommendations based on your stated interests
    recommendations = {
        "python": "Focus on backend frameworks like Node.js or Python-based automation.",
        "ml": "Review deep learning and computer vision principles for your final year project.",
        "sql": "Practice complex queries for your engineering college portal database.",
        "cybersecurity": "Look into MSc requirements in Germany as you planned."
    }

    suggestions = [recommendations.get(skill, "Research advanced certifications for this skill.")
                   for skill in missing_skills]

    return missing_skills, suggestions

In [54]:
from typing import TypedDict, List

# Define the data structure
class AgentState(TypedDict):
    query: str
    resume_text: str
    skills: List[str]
    jobs: List[dict]
    matched_jobs: List[dict]
    missing_skills: List[str]  # Required for gap analysis
    learning_path: List[str]   # Required for recommendations

# Helper: Logic for gap analysis
def analyze_skills_gap(user_skills, job_skills):
    user_set = set(s.lower() for s in user_skills)
    job_set = set(s.lower() for s in job_skills)
    missing = list(job_set - user_set)

    # Recommendations tailored to your projects (Web Portal/AI Agent)
    recs = {
        "python": "Master decorators and asyncio for your AI Job Agent.",
        "sql": "Study Window Functions for your Engineering College Portal database.",
        "ml": "Review Computer Vision for your final year project ideas.",
        "cybersecurity": "Research German MSc requirements as planned."
    }
    tips = [recs.get(s, "Search for advanced certifications in this skill.") for s in missing]
    return missing, tips

In [55]:
# 1. Extraction Node
def extract_skills_node(state):
    # Simplified logic based on your previous code
    skills_list = ["python", "java", "ml", "ai", "sql", "data science"]
    found = [s for s in skills_list if s in state["resume_text"].lower()]
    return {"skills": found}

# 2. Search Node
def search_jobs_node(state):
    # Mock data based on your current unified_job_search logic
    query = state["query"]
    jobs = [
        {"title": f"{query} Engineer", "company": "Tech Corp", "skills": ["python", "ml", "sql"]}
    ]
    return {"jobs": jobs}

# 3. Match Node
def match_jobs_node(state):
    user_skills = set(state["skills"])
    matched = []
    for job in state["jobs"]:
        score = len(user_skills.intersection(set(job["skills"])))
        if score > 0:
            job["match_score"] = score
            matched.append(job)
    return {"matched_jobs": sorted(matched, key=lambda x: x["match_score"], reverse=True)}

# 4. Gap Analysis Node (New)
def analyze_gap_node(state):
    if not state["matched_jobs"]:
        return {"missing_skills": [], "learning_path": []}

    top_job_skills = state["matched_jobs"][0].get("skills", [])
    missing, tips = analyze_skills_gap(state["skills"], top_job_skills)
    return {"missing_skills": missing, "learning_path": tips}

In [56]:
from langgraph.graph import StateGraph

builder = StateGraph(AgentState)

# Register the nodes
builder.add_node("extract_skills", extract_skills_node)
builder.add_node("search_jobs", search_jobs_node)
builder.add_node("match_jobs", match_jobs_node)
builder.add_node("analyze_gap", analyze_gap_node)

# Connect the nodes
builder.set_entry_point("extract_skills")
builder.add_edge("extract_skills", "search_jobs")
builder.add_edge("search_jobs", "match_jobs")
builder.add_edge("match_jobs", "analyze_gap")
builder.set_finish_point("analyze_gap")

graph = builder.compile()
print("Graph compiled successfully!")

Graph compiled successfully!


In [57]:
inputs = {
    "query": "Data Scientist",
    "resume_text": "I am a 6th sem student skilled in Python and SQL."
}

result = graph.invoke(inputs)

print("Missing Skills:", result["missing_skills"])
print("Learning Path:", result["learning_path"])

Missing Skills: ['ml']
Learning Path: ['Review Computer Vision for your final year project ideas.']


In [58]:
def tailor_resume_node(state):
    """Generates a tailored resume snippet using Gemini."""
    from google import genai
    client = genai.Client()

    # We take the top matched job and the extracted skills
    top_job = state["matched_jobs"][0] if state["matched_jobs"] else {}

    prompt = f"""
    You are an expert career coach. Rewrite the professional summary and 'Projects' section
    of this resume to better align with the following job title: {top_job.get('title', 'Software Engineer')}.

    Original Resume Content: {state['resume_text'][:2000]}
    Focus Skills: {', '.join(state['skills'])}
    Missing Skills to Address: {', '.join(state['missing_skills'])}

    Instructions:
    1. Highlight existing experience that matches the job.
    2. If the user has a 'College Portal' or 'AI Agent' project, emphasize the technical stack used (Node.js, SQLite, LangChain).
    3. Keep it professional and concise.
    """

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
    )

    return {"tailored_resume": response.text}

In [59]:
class AgentState(TypedDict):
    query: str
    resume_text: str
    skills: List[str]
    jobs: List[dict]
    matched_jobs: List[dict]
    missing_skills: List[str]
    learning_path: List[str]
    tailored_resume: str # New field for the output

builder = StateGraph(AgentState)
# ... add previous nodes (extract, search, match, analyze_gap) ...
builder.add_node("tailor_resume", tailor_resume_node)

# Update edges
builder.add_edge("analyze_gap", "tailor_resume")
builder.set_finish_point("tailor_resume")

In [60]:
def linkedin_optimizer(skills, role):
    suggestions = []

    # Headline suggestion
    suggestions.append(f"Update headline to: '{role} | {', '.join(skills[:3])}'")

    # About section
    suggestions.append("Write a strong 'About' section highlighting your skills and projects")

    # Skills section
    suggestions.append("Add relevant skills like: " + ", ".join(skills))

    # Projects
    suggestions.append("Add projects related to your domain (e.g., ML, Data Science projects)")

    # Networking
    suggestions.append("Connect with professionals in your domain and engage with posts")

    return suggestions

In [61]:
skills = ["python", "ml", "sql"]
role = "Data Scientist"

print(linkedin_optimizer(skills, role))

["Update headline to: 'Data Scientist | python, ml, sql'", "Write a strong 'About' section highlighting your skills and projects", 'Add relevant skills like: python, ml, sql', 'Add projects related to your domain (e.g., ML, Data Science projects)', 'Connect with professionals in your domain and engage with posts']


In [62]:
def extract_skills(resume):
    # simple example logic
    skills = ["Python", "Machine Learning", "SQL"]
    return skills

In [63]:
resume = """
I am a Computer Science student skilled in Python, Machine Learning, SQL, and Data Analysis.
I have worked on projects like Fake News Detection.
"""

In [64]:
resume = "Your resume text here"
query = "Data Scientist"

skills = extract_skills(resume)

print("\n--- LINKEDIN OPTIMIZATION ---")
for tip in linkedin_optimizer(skills, query):
    print("-", tip)


--- LINKEDIN OPTIMIZATION ---
- Update headline to: 'Data Scientist | Python, Machine Learning, SQL'
- Write a strong 'About' section highlighting your skills and projects
- Add relevant skills like: Python, Machine Learning, SQL
- Add projects related to your domain (e.g., ML, Data Science projects)
- Connect with professionals in your domain and engage with posts


In [65]:
def linkedin_score(skills):
    return len(skills) * 10